# ISK break-even sensitivity

This notebook sweeps realistic return and time-horizon ranges to find where ISK starts to dominate fondkonto, given the liquidation tax paid up front and the existing balance already inside the ISK.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

candidate_dirs = [Path.cwd(), Path.cwd() / 'copilot_analysis']
analysis_dir = next(path for path in candidate_dirs if (path / 'fund_analysis.py').exists())
if str(analysis_dir) not in sys.path:
    sys.path.append(str(analysis_dir))

from fund_analysis import DEFAULT_PORTFOLIO, TaxAssumptions, compare_accounts_grid

In [ ]:
annual_returns = np.arange(0.02, 0.251, 0.01)
years = np.arange(1, 41)
tax = TaxAssumptions(capital_gains_tax_rate=0.30, isk_tax_rate=0.01065, isk_tax_free_amount=300000.0)

grid = compare_accounts_grid(
    market_value=DEFAULT_PORTFOLIO['market_value'],
    cost_basis=DEFAULT_PORTFOLIO['cost_basis'],
    existing_isk_value=DEFAULT_PORTFOLIO['existing_isk_value'],
    annual_returns=annual_returns,
    years=years,
    tax=tax,
)
grid.head()

,annual_return,years,investable_isk_capital,tax_paid_now,isk_tax_free_amount,taxable_isk_base_marginal,annual_isk_tax,taxable_isk_base_total_account,annual_isk_tax_total_account,isk_terminal_value,fondkonto_terminal_value,difference_isk_minus_fondkonto
0,0.02,1.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,601506.313967,606763.566000,-5257.252033
1,0.02,2.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,607316.673665,617809.388760,-10492.715095
2,0.02,3.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,613370.174569,629076.127975,-15705.953407
3,0.02,4.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,619671.679502,640568.201975,-20896.522472
4,0.02,5.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,626226.148547,652290.117454,-26063.968908


In [ ]:
heatmap_data = grid.pivot(index='annual_return', columns='years', values='difference_isk_minus_fondkonto')
max_abs = float(np.nanmax(np.abs(heatmap_data.values)))

fig = px.imshow(
    heatmap_data,
    aspect='auto',
    origin='lower',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    range_color=[-max_abs, max_abs],
    title='ISK minus fondkonto across return and horizon',
    labels={'x': 'Years', 'y': 'Annual return', 'color': 'SEK advantage'}
)
fig.show()

In [4]:
break_even = (
    grid.sort_values(['annual_return', 'years'])
    .assign(is_break_even=lambda frame: frame['difference_isk_minus_fondkonto'] >= 0)
    .loc[lambda frame: frame['is_break_even']]
    .groupby('annual_return', as_index=False)['years']
    .min()
    .rename(columns={'years': 'first_break_even_year'})
)
break_even

,annual_return,first_break_even_year
0,0.05,32.0
1,0.06,22.0
2,0.07,15.0
3,0.08,11.0
4,0.09,7.0
5,0.10,5.0
6,0.11,3.0
7,0.12,1.0
8,0.13,1.0
9,0.14,1.0


In [5]:
fig = px.line(
    break_even,
    x='annual_return',
    y='first_break_even_year',
    markers=True,
    title='Break-even horizon by annual return assumption'
)
fig.show()

## Next extensions

Add capital ranges, alternative tax regimes, and inflation later if you want to stress-test the conclusion instead of just plotting the base grid.